# 嵌入

上一章，词袋模型成功完成了情感分类任务，但它的输入是一个稀疏的单热向量：词表有多大，向量就有多长，而且其中绝大多数都是 0。在实际应用中，词表动辄几万甚至几十万，这样的输入会带来两个严重问题：**内存消耗巨大**。每个词袋向量几乎都是零，存储和计算严重浪费。

## 向量压缩

是否可以参考数据压缩技术，把每个单热编码向量压缩到一个相对较短的向量？比如：如果我们的词表包括 10000 个单词。因此每个单词的单热编码向量的长度也是 10000。我们想把它压缩到 128。

解决方法很直观：用一个线性层把高维稀疏向量压缩成低维稠密向量。

比如词表有 10000 个词，我们想把每个词压缩到 128 维。线性层的权重矩阵形状是 `(128, 10000)`，每一列对应词表中的一个词。

关键发现：单热编码向量与这个权重矩阵做矩阵乘法，结果正好等于取出权重矩阵中那个 1 所在的列。换句话说，**单热编码的矩阵乘法，本质上就是查表**。

## 嵌入：用查表代替矩阵乘法

既然结果等价，我们可以跳过矩阵乘法，直接用词的**索引编码**去查权重矩阵对应的行。这就是**嵌入**（Embedding）。

```{figure} images/embedding.png
:align: center
:width: 640px
**图例：词袋嵌入示意**

* $\text{A B C}$：（输入）文字；
* $(1, 4, 8)$：（词袋）索引编码；
* $[0, 1, 0, 0, 1, 0, 0, 0, 1, 0]$：（词袋）单热编码
* $w$：（嵌入）权重。

```

对于词袋（多个词），我们取出每个词对应的嵌入向量，然后对它们**求平均**，得到整条影评的一个固定维度的表示。求平均的好处是：无论影评有多长，最终的表示维度都一样，而且对词的顺序不敏感（与词袋的特性一致）。

In [1]:
import csv
import re
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### IMDB 数据集

与上一章相比，`IMDBDataset` 有一处关键变化：

**特征从单热向量改为索引列表**。上一章的 `train_features` 是完整的单热编码向量（长度等于词表大小）；这一章直接存去重后的索引列表（如 `[3, 17, 42]`），让嵌入层在前向传播时实时查表。

这样做的好处是内存占用大幅减少——词表有 150 个词，每条影评平均出现 20 个不重复词，索引列表只需存 20 个整数，而单热向量需要存 150 个浮点数。

In [8]:
class IMDBDataset(Dataset):

    def __init__(self, filename):
        self.filename = filename
        super().__init__()

    def load(self):
        self.reviews = []
        self.sentiments = []
        with open(self.filename, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                self.reviews.append(self.clean_text(row[0].lower()).split())
                self.sentiments.append([0 if row[1] == "negative" else 1])

        self.vocabulary = sorted(set(word for line in self.reviews for word in line))
        self.word2index = {word: index for index, word in enumerate(self.vocabulary)}
        self.index2word = {index: word for index, word in enumerate(self.vocabulary)}
        self.tokens = [[self.word2index[word] for word in line if word in self.word2index] for line in self.reviews]

        # 词袋索引编码
        self.train_features = [list(set(tokens)) for tokens in self.tokens[:-100]]
        self.test_features = [list(set(tokens)) for tokens in self.tokens[-100:]]
        self.train_labels = self.sentiments[:-100]
        self.test_labels = self.sentiments[-100:]

    @staticmethod
    def clean_text(text):
        text = re.sub(r'<[^>]+>', '', text)
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
        return text

    def encode(self, text):
        words = self.clean_text(text.lower()).split()
        return [self.word2index[word] for word in words if word in self.word2index]

    def decode(self, tokens):
        return " ".join([self.index2word[index] for index in tokens])

    def estimate(self, predictions):
        labels = np.array(self.labels)
        correct = (np.abs(predictions.data - labels) < 0.5).sum()
        return correct / len(labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 嵌入层

**嵌入层**（Embedding Layer）将词的索引直接映射为低维稠密向量，跳过了单热编码和矩阵乘法。

**前向传播**：
- 输入是一条影评的词袋索引列表，例如 `[3, 17, 42]`；
- 用这些索引直接从权重矩阵中取出对应的行，得到形状 `(in_size, embedding_size)` 的矩阵；
- 对所有词的嵌入向量**求平均**（axis=0），得到形状 `(embedding_size,)` 的向量，作为这条影评的表示。

**梯度计算**：
- 梯度只需累加到被取出的那几行权重上，其他行不参与本次更新。

In [11]:
class Embedding(Layer):

    def __init__(self, vocabulary_size, embedding_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.embedding_size = embedding_size
        self.weight = Tensor(np.random.randn(vocabulary_size, embedding_size) * np.sqrt(2 / vocabulary_size))

    def forward(self, x: Tensor):
        p = Tensor(np.sum(self.weight.data[x.data], axis=1))

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

    def __repr__(self):
        return f'Embedding[weight{self.weight.data.shape}; vocabulary={self.vocabulary_size}; embedding={self.embedding_size}]'

### Sigmoid 激活函数层

In [12]:
class Sigmoid(Layer):

    def __init__(self, clip_range=(-100, 100)):
        super().__init__()
        self.clip_range = clip_range

    def forward(self, x: Tensor):
        z = np.clip(x.data, self.clip_range[0], self.clip_range[1])
        a = Tensor(1 / (1 + np.exp(-z)))

        def gradient_fn():
            x.grad += a.grad * a.data * (1 - a.data)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 二元交叉熵损失函数

In [13]:
class BCELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        clipped = np.clip(p.data, 1e-7, 1 - 1e-7)
        bce = Tensor(0 - np.sum(y.data * np.log(clipped) + (1 - y.data) * np.log(1 - clipped)) / len(y.data))

        def gradient_fn():
            p.grad += (clipped - y.data) / (clipped * (1 - clipped)) / len(y.data)

        bce.gradient_fn = gradient_fn
        bce.parents = {p}
        return bce

### 随机梯度下降优化器

In [14]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

训练流程与上一章相同。

测试流程有所调整：由于每条影评的词袋长度不同，无法整批送入，改为逐条推理后拼接结果。

In [15]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        # 词袋长度不同，逐条推理后拼接
        predictions = []
        for i in range(len(dataset)):
            feature, _ = dataset[i]
            predictions.append(self.layer(feature).data)

        predictions = Tensor(np.reshape(predictions, (len(predictions), -1)))
        loss = self.loss_fn(predictions, Tensor(dataset.labels))
        return predictions, loss

## 训练

### 超参数：学习率

In [16]:
LEARNING_RATE = 0.1

### 超参数：轮次

In [17]:
EPOCHS = 100

### 建模

模型结构：

- **嵌入层**：`Embedding(150, 32)`，按词袋索引查表后平均，得到 32 维的影评向量；
- **线性层**：`Linear(32, 1)`，把 32 维特征压缩成一个标量；
- **Sigmoid激活函数**：映射到 (0, 1) 区间，作为喜欢的概率。

In [18]:
dataset = IMDBDataset('tinyimdb.csv')

layer = Sequential([
    Embedding(len(dataset.vocabulary), 32),
    Linear(32, 1),
    Sigmoid()
])
loss_fn = BCELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Embedding[weight(150, 32); vocabulary=150; embedding=32]
Linear[weight(1, 32); bias(1,)]
Sigmoid[]


### 迭代

In [19]:
model.train(dataset, EPOCHS)

## 验证

### 测试

训练完毕，在测试集上评估准确率：

In [20]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 100.00%


同样达到 100% 准确率，验证了嵌入层可以完全替代单热编码词袋，而且在大词表场景下效率会更高。

### 嵌入权重相似性

和线性层不同，嵌入层的权重和单词有一一对应的关系：每个单词都有独属于自己的权重。

让我们观察一下权重最相似，和最不相似的词之间有什么特点。

In [21]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def find_similar_words(word, top_n=5):
    if word not in dataset.word2index:
        return

    idx = dataset.word2index[word]
    target_vec = layer.layers[0].weight.data[idx]

    similarities = []
    for i, w in enumerate(dataset.vocabulary):
        if i == idx:
            continue
        sim = cosine_similarity(target_vec, layer.layers[0].weight.data[i])
        similarities.append((sim, w))

    similarities.sort(reverse=True)
    print(f'Similar to "{word}":')
    for sim, w in similarities[:top_n]:
        print(f'  {w:20s}  {sim:.4f}')
    print(f'Not silimar to "{word}": ')
    for sim, w in similarities[-top_n:][::-1]:
        print(f'  {w:20s}  {sim:.4f}')


find_similar_words('good')

Similar to "good":
  beautiful             0.5937
  adored                0.5665
  at                    0.5197
  screenplay            0.5155
  highly                0.4622
Not silimar to "good": 
  awful                 -0.4254
  dreadful              -0.3953
  hated                 -0.3909
  we                    -0.3805
  avoid                 -0.3725


我们可以看到，和 good 的嵌入权重相似的词，也恰恰是它的相似词；而嵌入权重差别最大的词，也是和它相反的词。

这也非常形象地展示了网络模型中的权重学到了什么。

``💡 嵌入权重是通过情感分类任务训练出来的，所以它学到的是情感上的相似，而不全然是语义上的相似。``

## 课后作业

试验不同的单词，看看网络模型学习到的相似词和相反词都是哪些。